In [ ]:
# Standard library
import os
import warnings
from pathlib import Path

# Third-party scientific stack
import pandas as pd
import numpy as np
from tqdm import tqdm
from skbio.diversity.alpha import (
    shannon,
    gini_index,
    pielou_e,
    simpson,
    chao1,
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import plotly.graph_objects as go

# Scientific computing
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.sparse import csr_matrix
from scipy.stats import iqr

from sklearn.manifold import MDS

# Local project utilities
from utils.Processing import (
    annotate_values,
    run_stats_table_for_metric,
)

# Global settings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
os.environ["OMP_NUM_THREADS"] = "8"

# Matplotlib configuration
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "Arial"

# Discovery cohort TCR overview

(Manuscript figures: Fig. 1A; Fig. S1)

In [ ]:
# Base directory of the files
base_dir = Path("..").resolve()

# Directories containing the processed files
processed_dir = base_dir / "2_processed_data" / "0_discovery_data"

# Output directories for the figures and tables
figures_dir = base_dir / "3_figures" / "Figures"
figures_dir.mkdir(exist_ok=True)
tables_dir = base_dir / "3_figures" / "Tables"
tables_dir.mkdir(exist_ok=True)

### Fig. 1A: TCR richness & diversity

In [ ]:
# Define function for DE25 calculation
def dominance_25_percent(data, count_col="clonecount_celltype"):
    """
    Calculate DE25: proportion of clonotypes required to reach 25% cumulative frequency.

    Parameters
    ----------
    data : pd.DataFrame
        Input dataframe containing clone counts.
    count_col : str, default="clonecount_celltype"
        Column name with absolute clone counts.

    Returns
    -------
    float
        Ratio of clonotypes of top expanded clones (0–1) required to reach 25% of total frequency.
    """
    res = data.sort_values(by=count_col, ascending=False).reset_index(drop=True)
    total_sum = res[count_col].sum()
    res["temp_freq"] = res[count_col] / total_sum
    res["cumulative_sum"] = res["temp_freq"].cumsum()

    first_index = res[res["cumulative_sum"] >= 0.25].index[0]
    return (first_index + 1) / len(res)


# Keep only TRA and TRB chains
full_df = pd.read_csv(processed_dir / "0_final_data_parsed_annotated.csv")
tcr_df = full_df[full_df["chain"].isin(["TRA", "TRB"])]

patients = tcr_df["patient"].unique()
chains = tcr_df["chain"].unique()
celltypes = tcr_df["cell_type"].unique()


####################################
### Alpha diversity calculations ###
####################################
results = []

for patient in tqdm(patients, desc="Processing patients"):
    for chain in chains:
        for celltype in celltypes:
            subset = tcr_df[
                (tcr_df["patient"] == patient)
                & (tcr_df["chain"] == chain)
                & (tcr_df["cell_type"] == celltype)
            ]

            if not subset.empty:
                clone_counts = subset["clonecount_celltype"]

                results.append(
                    {
                        "Patient": patient,
                        "Chain": chain,
                        "Cell_type": celltype,
                        "Richness": len(clone_counts),
                        "Shannon_index": shannon(clone_counts, base=2),
                        "Pielou_evenness": pielou_e(clone_counts),
                        "Gini_Simpson": simpson(clone_counts),
                        "Gini_coefficient": gini_index(clone_counts),
                        "DE25": dominance_25_percent(subset),
                    }
                )


div_df = pd.DataFrame(results)
div_df.to_csv(processed_dir / "2_diversity_calc.csv", index=False)

print("Diversity metrics saved successfully")

# Define condition labels
div_df["Condition"] = (
    div_df["Chain"].astype(str) + "-" + div_df["Cell_type"].astype(str)
)

# Get the richness results separately
df_richness = div_df[["Patient", "Condition", "Richness"]].copy()
df_richness["x_value"] = "Full cohort"

# Diversity metrics
metrics_to_plot = ["Gini_Simpson", "Gini_coefficient", "Pielou_evenness", "DE25"]
df_long = div_df.melt(
    id_vars=["Patient", "Condition"],
    value_vars=metrics_to_plot,
    var_name="Metric",
    value_name="Value",
)

# Set the condition column as a categorical variable with a specified order
conditions = [
    "TRA-CD4",
    "TRB-CD4",
    "TRA-CD8",
    "TRB-CD8",
]

df_long["Condition"] = pd.Categorical(
    df_long["Condition"],
    categories=conditions,
    ordered=True,
)

df_richness["Condition"] = pd.Categorical(
    df_richness["Condition"],
    categories=conditions,
    ordered=True,
)

# TCR numbers and overview
total_sequences = len(tcr_df)

sequence_summary = {
    "Total sequences": total_sequences,
    "TRA sequences": (tcr_df["chain"] == "TRA").sum(),
    "TRB sequences": (tcr_df["chain"] == "TRB").sum(),
    "CD4 sequences": (tcr_df["cell_type"] == "CD4").sum(),
    "CD8 sequences": (tcr_df["cell_type"] == "CD8").sum(),
}

for k, v in sequence_summary.items():
    print(f"{k}: {v:,}")


# Calculate median diversity for each condition and metric
median_diversity = (
    df_long[df_long["Metric"].isin(metrics_to_plot)]
    .groupby(["Condition", "Metric"])["Value"]
    .median()
    .reset_index()
    .rename(columns={"Value": "Median"})
)

# Add richness median to the diversity metrics
df_richness_median = df_richness.groupby("Condition")["Richness"].median().reset_index()
df_richness_median["Metric"] = "Richness"
df_richness_median.rename(columns={"Richness": "Median"}, inplace=True)

median_diversity = pd.concat(
    [median_diversity, df_richness_median], ignore_index=True
).pivot(index="Metric", columns="Condition", values="Median")

print(median_diversity)


#####################################################
### Statistcal testing between metrics and groups ###
#####################################################
results = []

res_rich = run_stats_table_for_metric(
    df_richness.rename(columns={"Richness": "Value"}),
    metric_label="Richness",
    value_col="Value",
)
results.append(res_rich)

for metric in metrics_to_plot:
    df_metric = df_long[df_long["Metric"] == metric]
    res_metric = run_stats_table_for_metric(
        df_metric,
        metric_label=metric,
        value_col="Value",
    )
    results.append(res_metric)

all_stats = pd.concat(results, ignore_index=True)

all_stats_table = all_stats[["Metric", "Group1", "Group2", "p_value_adj"]].copy()
all_stats_table["Significance"] = all_stats_table["p_value_adj"].apply(
    lambda p: ("***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns")
)
all_stats_table.sort_values(by="p_value_adj", ascending=True, inplace=True)

all_stats_table.to_csv(tables_dir / "diversity_stats.csv", index=False)

In [ ]:
#########################
### Figure generation ###
#########################

palette = {
    "TRA-CD4": "#9ecae1",
    "TRB-CD4": "#3182bd",
    "TRA-CD8": "#fdae6b",
    "TRB-CD8": "#e6550d",
}

add_annotations = False  # set True to draw brackets

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 8),
    gridspec_kw={"width_ratios": [1, 3], "wspace": 0.1},
    sharey=False,
)

# Panel 1: Richness boxplots
sns.boxplot(
    data=df_richness,
    x="x_value",
    y="Richness",
    hue="Condition",
    hue_order=conditions,
    palette=palette,
    fliersize=0,
    width=0.8,
    legend=False,
    linewidth=1.5,
    boxprops=dict(edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    ax=axes[0],
)

axes[0].set_ylabel("")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", labelsize=14)
axes[0].tick_params(axis="y", labelsize=12)
axes[0].set_title("TCR Richness", fontsize=18, fontweight="bold", pad=15)
sns.despine(ax=axes[0])

if add_annotations:
    annotate_values(
        axes[0],
        df_richness.rename(columns={"Richness": "Value"}),
        value_col="Value",
    )

# Panel 2: Diversity metric boxplots
ax = axes[1]
sns.boxplot(
    data=df_long,
    x="Metric",
    y="Value",
    hue="Condition",
    hue_order=conditions,
    palette=palette,
    fliersize=0,
    width=0.8,
    linewidth=1.5,
    boxprops=dict(edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    ax=ax,
    dodge=True,
)

ax.set_ylabel("")
ax.set_xlabel("")
ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=12)
ax.set_title("Diversity Metrics", fontsize=18, fontweight="bold", pad=15)
ax.set_xticklabels([m.replace("_", " ") for m in metrics_to_plot], size=15)
sns.despine(ax=ax)

if add_annotations:
    for i, metric in enumerate(metrics_to_plot):
        df_metric = df_long[df_long["Metric"] == metric]
        annotate_values(
            ax,
            df_metric,
            value_col="Value",
            x_base=i,
        )

# Zoom in on the DE25 values since they are so small
axins = inset_axes(
    ax,
    width="45%",
    height="60%",
    bbox_to_anchor=(0.5, -0.185, 0.5, 0.5),
    bbox_transform=ax.transAxes,
)
sns.boxplot(
    data=df_long[df_long["Metric"] == "DE25"],
    x="Metric",
    y="Value",
    hue="Condition",
    hue_order=conditions,
    palette=palette,
    fliersize=0,
    width=0.8,
    linewidth=1.5,
    boxprops=dict(edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    ax=axins,
    dodge=True,
    legend=False,
)

axins.set_title("", fontsize=10, weight="bold")
axins.set_xlabel("")
axins.set_ylabel("")
axins.set_ylim(-0.01, 0.08)
axins.tick_params(axis="x", labelsize=0)
axins.tick_params(axis="y", labelsize=10)
sns.despine(ax=axins)

if add_annotations:
    df_de25 = df_long[df_long["Metric"] == "DE25"]
    annotate_values(axins, df_de25, value_col="Value")

# Legend
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles[: len(palette)],
    labels[: len(palette)],
    title="",
    loc="lower center",
    ncol=4,
    fontsize=12,
    bbox_to_anchor=(0.49, 0.06),
)
ax.get_legend().remove()

plt.tight_layout()
plt.subplots_adjust(wspace=0.25, bottom=0.18)
plt.savefig(figures_dir / "1A_diversity.pdf", bbox_inches="tight")
plt.show()

### Fig. S1A: Beta diversity for the TCR repertoire

In [ ]:
# Load processed TCR data
full_df = pd.read_csv(processed_dir / "0_final_data_parsed_annotated.csv")

print("\nTransform to count matrix...")
count_df = full_df.pivot_table(
    index="patient", columns="full_tcr", aggfunc="size", fill_value=0
)

print("\nConvert to sparse matrix...")
X_sparse = csr_matrix(count_df.values, dtype=np.float32)

print("\nComputing relative abundance...")
row_sums = np.array(X_sparse.sum(axis=1)).flatten()
row_indices, col_indices = X_sparse.nonzero()
X_sparse.data /= row_sums[row_indices]

print("\nConvert to dense and compute distances...")
X_dense = X_sparse.astype(np.float32).toarray()
dist_vector = pdist(X_dense, metric="braycurtis")
dist_matrix = squareform(dist_vector)

# Summary statistics
mean_dist = np.mean(dist_vector)
median_dist = np.median(dist_vector)
std_dist = np.std(dist_vector)
q25, q75 = np.percentile(dist_vector, [25, 75])
min_dist, max_dist = dist_vector.min(), dist_vector.max()

print(f"Mean Bray-Curtis distance: {mean_dist:.2f}")
print(f"Median Bray-Curtis distance: {median_dist:.2f} (IQR {q25:.2f}-{q75:.2f})")
print(f"Std: {std_dist:.2f}, Min: {min_dist:.2f}, Max: {max_dist:.2f}")

# Non-metric MDS
nmds = MDS(
    n_components=2,
    dissimilarity="precomputed",
    metric=False,
    random_state=42,
    n_init=4,
    max_iter=300,
)
nmds_coords = nmds.fit_transform(dist_matrix)

In [ ]:
#########################
### Figure generation ###
#########################

plt.figure(figsize=(7, 7))
plt.scatter(
    nmds_coords[:, 0],
    nmds_coords[:, 1],
    color="#1A1A19",
    s=60,
    edgecolors="white",
    alpha=0.8,
)
plt.title("nMDS of Full TCR Repertoires", fontsize=16, fontweight="bold")
plt.xlabel("nMDS1")
plt.ylabel("nMDS2")
plt.savefig(figures_dir / "S1A_beta_diversity_TCR.pdf", bbox_inches="tight")
plt.show()

### Fig. S1B: Epitope predictions overview

In [ ]:
tcr_data = pd.read_csv(processed_dir / "0_final_data_parsed_annotated.csv")

###################################
### Epitope prediction overview ###
###################################

print(f"\nThe distribution of TCR chains: {tcr_data['chain'].value_counts()}")
print(f"\nThe distribution of TCR cell types: {tcr_data['cell_type'].value_counts()}")

tcr_epitopes = tcr_data[tcr_data["Score"] >= 0.2]
print(f"\nThere are {len(tcr_epitopes)} TCRs with a confident prediction.")
print(
    f"This includes {tcr_epitopes['Epitope'].nunique()} unique epitopes from {tcr_epitopes['Epitope Antigen'].nunique()} unique antigens."
)
print(
    f"\nThe top most abundant species are: {tcr_epitopes['Epitope Species'].value_counts().head(8)}"
)

tcr_epitopes_bact = tcr_epitopes[
    (tcr_epitopes["epitope_groups"] == "Microbial")
    & (tcr_epitopes["Epitope Species"] != "Triticum aestivum")
    & (tcr_epitopes["Epitope Species"] != "Plasmodium falciparum")
]
print(
    f"\nThere are {len(tcr_epitopes_bact)} TCRs predicted against bacterial antigens."
)
print(f"These include: {tcr_epitopes_bact['Epitope'].value_counts()}")

print(
    f"The bacterial species are linked to these epitopes and species: {tcr_epitopes_bact[['Epitope', 'Epitope Antigen', 'Epitope Species', 'epitope_groups']].drop_duplicates()}"
)


# To avoid errors
tcr_epitopes["Epitope Species"] = tcr_epitopes["Epitope Species"].replace(
    {"Synthetic": "Synthetic_"}
)


top_parents = tcr_epitopes["epitope_groups"].dropna().unique().tolist()
child_nodes = tcr_epitopes["Epitope Species"].dropna().unique().tolist()
labels = top_parents + child_nodes

parents = [""] * len(top_parents)

species_to_group = (
    tcr_epitopes[["Epitope Species", "epitope_groups"]]
    .dropna()
    .drop_duplicates()
    .set_index("Epitope Species")["epitope_groups"]
    .to_dict()
)

parents.extend([species_to_group[sp] for sp in child_nodes])

dict_groups = tcr_epitopes["epitope_groups"].value_counts().to_dict()
dict_species = tcr_epitopes["Epitope Species"].value_counts().to_dict()

values = [dict_groups[p] for p in top_parents] + [
    dict_species[ch] for ch in child_nodes
]


############################################
### Figure generation: Sunburst epitopes ###
############################################

color_map = {
    "Viral": "#636EFA",
    "Human": "#EF553B",
    "Microbial": "#00CC96",
    "Synthetic": "#AB63FA",
}

colors = []

for lab in labels:
    if lab in color_map:
        colors.append(color_map[lab])
    else:
        parent = species_to_group.get(lab, None)
        if parent and parent in color_map:
            colors.append(color_map[parent])
        else:
            colors.append("lightgrey")


fig = go.Figure(
    go.Sunburst(
        labels=labels,
        parents=parents,
        values=values,
        branchvalues="total",
        marker=dict(colors=colors),
        insidetextorientation="radial",
    )
)
fig.update_layout(margin=dict(t=5, l=5, r=5, b=5), title_text="", title_x=0.5)
fig.write_image(figures_dir / "S1B_sunburst_noepitopes.pdf")
fig.show()


####################################
### Figure generation: Pie chart ###
####################################

print(len(tcr_data))
print(len(tcr_data[tcr_data["Score"] >= 0.2]))

print(((len(tcr_data[tcr_data["Score"] >= 0.2])) / (len(tcr_data))) * 100)

main_counts = [5477366, 133605]
main_labels = ["Not (97.6%)", "Annotated (2.4%)"]
main_colors = ["#D3D3D3", "#1F78B4"]

plt.pie(
    main_counts,
    labels=["", ""],
    startangle=90,
    colors=main_colors,
    textprops={"fontsize": 25},
    wedgeprops={"linewidth": 2, "edgecolor": "white"},
)

plt.savefig(figures_dir / "S1B_small_pie.pdf", bbox_inches="tight")

# Discovery cohort microbiome overview

(Manuscript figures: Fig. 1B-D)

In [ ]:
meta = pd.read_excel(
    base_dir / "1_raw_data" / "0_discovery_data" / "AIRRWAS_metadata.xlsx"
)

meta["patient_id"] = meta["Subjectnr"].str.split(" ").str[1]

Patient_list = pd.read_csv(
    base_dir / "1_raw_data" / "0_discovery_data" / "Overlap_id_list.csv"
)["patient_id"]

Full_data = pd.read_csv(
    base_dir / "1_raw_data" / "0_discovery_data" / "Microbiome_full_names_taxa.csv"
)

Full_data_overlap = Full_data[Full_data["subject"].isin(Patient_list)]
Full_data_overlap["genus"] = Full_data_overlap["taxon_name"].str.split(" ").str[0]

asv_wide = Full_data_overlap.pivot_table(
    index="subject", columns="taxon_name", values="abundance", fill_value=0
)

### Fig 1B: Microbiome abundance
Plot the microbial abundance of the top 9 genera across all individual samples

In [ ]:
asv_rel = asv_wide.div(asv_wide.sum(axis=1), axis=0)
genus_names = asv_rel.columns.str.split(" ").str[0]
genus_abundance = asv_rel.groupby(genus_names, axis=1).sum()


top_n = 9
top_genera = genus_abundance.sum(axis=0).nlargest(top_n).index
df_selected_genera = genus_abundance.copy()
df_selected_genera["Other"] = 1 - df_selected_genera[top_genera].sum(axis=1)
df_selected_genera = df_selected_genera[top_genera.tolist() + ["Other"]]

taxon_order = top_genera.tolist() + ["Other"]

# Hierarchical clustering of samples based on Bray-Curtis distance
distance_matrix = cdist(
    df_selected_genera.values, df_selected_genera.values, metric="braycurtis"
)
Z = linkage(distance_matrix, method="complete")
sample_order = fcluster(Z, t=0.1, criterion="distance")
df_selected_genera_sorted = df_selected_genera.iloc[np.argsort(sample_order)]

# Color palette for the genera
custom_colors = [
    "#0173b2",
    "#de8f05",
    "#029e73",
    "#d55e00",
    "#56b4e9",
    "#ca9161",
    "#fbafe4",
    "#ece133",
    "#cc78bc",
    "#C3C0C0",
]
color_palette = dict(zip(df_selected_genera_sorted.columns, custom_colors))


fig, ax1 = plt.subplots(figsize=(8, 6))

df_selected_genera_sorted.plot(
    kind="bar",
    stacked=True,
    color=[color_palette[t] for t in df_selected_genera_sorted.columns],
    width=1,
    ax=ax1,
    edgecolor="none",
    legend=False,
    alpha=1,
)

ax1.set_xticks([])
ax1.set_ylim(0, 1)
ax1.set_xlabel("Samples", fontsize=12)
ax1.set_ylabel("Relative abundance", fontsize=12)
ax1.set_title("Genus-level Composition", fontsize=16, fontweight="bold")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.tick_params(axis="y", labelsize=10)

reversed_taxon_order = taxon_order[::-1]
reversed_custom_colors = [color_palette[taxon] for taxon in reversed_taxon_order]
handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color, markersize=10)
    for color in reversed_custom_colors
]

fig.legend(
    handles=handles,
    labels=reversed_taxon_order,
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False,
    fontsize=12,
    title="Dominant genus",
)

plt.tight_layout()
plt.savefig(figures_dir / "1B_micro_abundance.pdf", bbox_inches="tight")
plt.show()

### Fig 1C: Microbiome alpha diveristy

In [ ]:
def rarefy_counts(counts, depth, rng):
    """
    Rarefy a 1D array of integer counts to a fixed depth
    using sampling without replacement.
    """
    if counts.sum() < depth:
        raise ValueError("Sample has fewer reads than rarefaction depth")

    expanded = np.repeat(np.arange(len(counts)), counts)
    subsampled = rng.choice(expanded, depth, replace=False)
    return np.bincount(subsampled, minlength=len(counts))


depth = 25000
rng = np.random.default_rng(42)

# ensure integer counts
asv_table = asv_wide.astype(int)

# drop samples below depth
asv_filt = asv_table.loc[asv_table.sum(axis=1) >= depth]

# rarefy
rarefied_asv = asv_filt.apply(
    lambda row: rarefy_counts(row.values, depth, rng), axis=1, result_type="expand"
)

rarefied_asv.columns = asv_filt.columns

print(rarefied_asv.sum(axis=1).unique())

results = []

for sample in rarefied_asv.index:
    counts = rarefied_asv.loc[sample].values

    results.append(
        {
            "subject": sample,
            "Richness_observed": (counts > 0).sum(),
            "Chao1": chao1(counts),
            "Shannon": shannon(counts, base=2),
            "Simpson": simpson(counts),
            "Pielou_evenness": pielou_e(counts),
        }
    )

alpha_df = pd.DataFrame(results)

rarefied_rel = rarefied_asv.div(rarefied_asv.sum(axis=1), axis=0)
genus_names = rarefied_rel.columns.str.split(" ").str[0]
genus_abundance = rarefied_rel.groupby(genus_names, axis=1).sum()
dominant_genus = genus_abundance.idxmax(axis=1)
dominant_genus_values = genus_abundance.max(axis=1)

dominant_genus_df = pd.DataFrame(
    {
        "subject": dominant_genus.index,
        "dominant_genus": dominant_genus.values,
        "dominant_genus_value": dominant_genus_values.values,
    }
)

alpha_df = alpha_df.merge(
    dominant_genus_df[["subject", "dominant_genus", "dominant_genus_value"]],
    on="subject",
    how="left",
)

alpha_df["dom_genus_50"] = np.where(
    alpha_df["dominant_genus_value"] >= 0.50,
    alpha_df["dominant_genus"],
    "Other",
)
alpha_df.sort_values(by="dominant_genus_value", ascending=False).head(10)

metrics = ["Richness_observed", "Chao1", "Shannon", "Simpson", "Pielou_evenness"]

# Compute median, IQR and z-score for each metric

for metric in metrics:
    median_val = alpha_df[metric].median()
    iqr_val = iqr(alpha_df[metric])
    # Avoid division by zero if IQR=0
    if iqr_val == 0:
        iqr_val = 1e-9
    alpha_df[f"{metric}_zscore"] = (alpha_df[metric] - median_val) / iqr_val

# Identify outliers using 1.5 * IQR

for metric in metrics:
    Q1 = alpha_df[metric].quantile(0.25)
    Q3 = alpha_df[metric].quantile(0.75)
    IQR_val = Q3 - Q1
    lower = Q1 - 1.5 * IQR_val
    upper = Q3 + 1.5 * IQR_val
    alpha_df[f"{metric}_outlier"] = ~alpha_df[metric].between(lower, upper)

# Flag any sample that is an outlier in at least one metric
alpha_df["any_outlier"] = alpha_df[[f"{m}_outlier" for m in metrics]].any(axis=1)

# Compute composite deviation score
alpha_df["deviation_score"] = (
    alpha_df[[f"{m}_zscore" for m in metrics]].abs().sum(axis=1)
)

# Summary statistics per metric
summary_stats = {}
for metric in metrics:
    summary_stats[metric] = {
        "median": alpha_df[metric].median(),
        "mean": alpha_df[metric].mean(),
        "std": alpha_df[metric].std(),
        "min": alpha_df[metric].min(),
        "max": alpha_df[metric].max(),
        "n_outliers": alpha_df[f"{metric}_outlier"].sum(),
    }

# Count total number of samples flagged as outliers in any metric
total_any_outlier = alpha_df["any_outlier"].sum()

# Top deviating samples
top_deviation_samples = alpha_df.sort_values("deviation_score", ascending=False)[
    ["subject", "deviation_score"] + metrics
].head(10)

stat_df = pd.merge(
    top_deviation_samples,
    alpha_df[["subject", "dominant_genus", "dominant_genus_value", "dom_genus_50"]],
    on="subject",
    how="left",
)

# All samples that are outliers in any alpha metric
outlier_samples = alpha_df.loc[alpha_df["any_outlier"], "subject"].tolist()
print("Samples flagged as outliers in at least one metric:")
print(outlier_samples)

# Optionally, outliers per metric
for metric in metrics:
    metric_outliers = alpha_df.loc[alpha_df[f"{metric}_outlier"], "subject"].tolist()
    print(f"{metric} outliers: {metric_outliers}")

In [ ]:
# Define panels and metrics
panel_metrics = {
    "Richness": ["Chao1"],
    "Shannon": ["Shannon"],
    "Other Diversity": ["Simpson", "Pielou_evenness"],
}

metric_labels = {
    "Shannon": "Shannon Diversity",
    "Simpson": "Simpson Diversity",
    "Pielou_evenness": "Pielou Evenness",
    "Richness_observed": "Observed Richness",
    "Chao1": "Chao1 Richness",
}

palette = {
    "Bacteroides": "#0173b2",
    "Prevotella": "#de8f05",
    "Faecalibacterium": "#029e73",
    "Akkermansia": "#56b4e9",
    "Other": "#D8D6D6",
}

# Create 3 subplots
fig, axes = plt.subplots(
    1,
    len(panel_metrics),
    figsize=(12, 6),
    sharey=False,
    gridspec_kw={"width_ratios": [1, 1, 2], "wspace": 0.3},
)

for ax, (panel_name, metrics) in zip(axes, panel_metrics.items()):
    df_long = alpha_df.melt(
        id_vars=["subject", "dom_genus_50"],
        value_vars=metrics,
        var_name="Metric",
        value_name="Value",
    )
    df_long["Metric_label"] = df_long["Metric"].map(metric_labels)

    df_long = df_long.merge(
        alpha_df[["subject", "dom_genus_50"] + [f"{m}_outlier" for m in metrics]],
        on=["subject", "dom_genus_50"],
        how="left",
    )

    df_long["is_outlier"] = df_long.apply(
        lambda row: row[f"{row['Metric']}_outlier"],
        axis=1,
    )

    sns.boxplot(
        data=df_long,
        x="Metric_label",
        y="Value",
        color="#f2f2f2",
        width=0.6,
        linewidth=1.5,
        showfliers=False,
        fliersize=6,
        flierprops=dict(
            marker="o",
            markerfacecolor="white",
            markeredgecolor="black",
            markersize=7,
            linestyle="none",
        ),
        boxprops=dict(edgecolor="black"),
        whiskerprops=dict(color="black"),
        capprops=dict(color="black"),
        medianprops=dict(color="black", linewidth=2.5),
        ax=ax,
    )
    sns.stripplot(
        data=df_long[df_long["is_outlier"]],
        x="Metric_label",
        y="Value",
        hue="dom_genus_50",
        palette=palette,
        size=9,
        jitter=0.05,
        edgecolor="black",
        linewidth=1,
        ax=ax,
        legend=False,
    )
    ax.set_title(panel_name, fontsize=16, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(figures_dir / "1C_micro_alpha_diversity.pdf", bbox_inches="tight")
plt.show()

### Fig 1D: Microbiome beta diveristy

In [ ]:
rarefied_rel = rarefied_asv.div(rarefied_asv.sum(axis=1), axis=0)

# Bray-Curtis distance
distance_matrix = squareform(pdist(rarefied_rel.values, metric="braycurtis"))

# NMDS
mds = MDS(
    n_components=2,
    dissimilarity="precomputed",
    metric=False,
    random_state=42,
    n_init=8,
    max_iter=300,
)
nmds_coords = mds.fit_transform(distance_matrix)

nmds_df = pd.DataFrame(
    nmds_coords,
    index=rarefied_rel.index,
    columns=["NMDS1", "NMDS2"],
)

genus_names = rarefied_rel.columns.str.split(" ").str[0]
genus_abundance = rarefied_rel.groupby(genus_names, axis=1).sum()
dominant_genus = genus_abundance.idxmax(axis=1)
dominant_genus_values = genus_abundance.max(axis=1)

dominant_genus_df = pd.DataFrame(
    {
        "subject": genus_abundance.index,
        "dom_genus_50": np.where(
            dominant_genus_values >= 0.50,
            dominant_genus,
            "Other",
        ),
    }
)

dominant_genus_df = dominant_genus_df[["subject", "dom_genus_50"]]

dominant_genus_df["colors"] = dominant_genus_df["dom_genus_50"].map(color_palette)

plot_df = nmds_df.merge(dominant_genus_df, left_index=True, right_on="subject")

fig, ax2 = plt.subplots(figsize=(6, 6))

scatter = ax2.scatter(
    plot_df["NMDS1"],
    plot_df["NMDS2"],
    c=plot_df["colors"],
    s=100,
    alpha=1,
    edgecolors="white",
    linewidths=2,
)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.set_title(
    "NMDS of microbial community (ASV level)", fontsize=18, pad=20, fontweight="bold"
)
ax2.set_xlabel("NMDS1", fontsize=14)
ax2.set_ylabel("NMDS2", fontsize=14)
ax2.tick_params(axis="both", labelsize=10)


plt.tight_layout()
plt.savefig(figures_dir / "1D_micro_beta_diversity.pdf", bbox_inches="tight")
plt.show()